In [8]:
import os

import introns
from collections import defaultdict
from Bio import SeqIO
from Bio.Seq import Seq
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import poisson
from Bio.SeqUtils import GC
import random

In [2]:
path="/mnt/archive/euglena_genomy/"
genom_EG=path+"gracilis/gracilis_dbg2olc.fasta"
genom_EH=path+'hiemalis/hiemalis_rascaf.fasta'
genom_EL=path+'longa/longa_rascaf.fasta'
genom_bugtest = '/home/semik/projekty/noncanonical_introns/bugtest.fasta'

In [3]:
GCs=[]
for genom in [genom_bugtest, genom_EG, genom_EH, genom_EL]:
    listaGC=[]
    for record in SeqIO.parse(genom, "fasta"):
        listaGC.append(GC(record.seq))
    gc=sum(listaGC)/len(listaGC)
    GCs.append(gc)
meanGC=sum(GCs[1:])/3
print(meanGC)

50.48750886317854


In [ ]:
def mojeGC(sekw):
    count=0
    for i in sekw:
        if i in ["G", "C"]:
            count+=1
    return count/len(sekw)*100

for genom in [genom_bugtest, genom_EG, genom_EH, genom_EL][:1]:
    listadoGC=''
    for record in SeqIO.parse(genom, "fasta"):
        listadoGC+=record.seq
    print(GC(listadoGC)==mojeGC(listadoGC))

In [ ]:
length=60
at=(100-meanGC)/200
gc=meanGC/200
gc_intron, gc_exon = 52, 62

print(at, gc)
break

for zbior in range(1):
    i=1
    with open("generowane_introny_%s.fasta" %zbior, 'w') as test_fasta:
        test_fasta.write(";nazwa - nrzbioru;5-65;wersja=0;wersja konw;wersja niekonw\n")
        test_fasta.write(";seq - exon[-5:] intron[:25] intron[-25:] exon[:5]\n")
        while i<=1000000:
            seq="".join(np.random.choice(["A", "T", "C", "G"], 60, p=[at, at, gc, gc]))
            prev_seq, seqAAA, next_seq = seq[:5], seq[5:30]+"A"*10+seq[30:-5], seq[-5:]
            #print(len(seqAAA))
            scaffold_name = "0"
            prev_exon, next_exon = introns.Exon(scaffold_name=scaffold_name, scaffold_start=0, scaffold_end=5, sequence=prev_seq), introns.Exon(scaffold_name=scaffold_name, scaffold_start=65, scaffold_end=70, sequence=next_seq)
            created_intron = introns.Intron(scaffold_name=scaffold_name, scaffold_start=5, scaffold_end=65, sequence=seqAAA, prev_exon=prev_exon, next_exon=next_exon)
            created_intron.conventional_version()
            created_intron.nonconventional_version()
            test_fasta.write(">%s;%s-%s;0;%s;%s\n%s\n" %(zbior,created_intron.scaffold_start, created_intron.scaffold_end, created_intron.best_conv_var, created_intron.best_nonconv_var, seqAAA))
            i+=1

In [39]:
gc_intron, gc_exon = .52/2, .62/2
at_intron, at_exon = (1-.52)/2, (1-.62)/2


rep_lengths = np.arange(0,11)
repetition_probs = np.array([0.22109991819985447, 0.25550569211509866, 0.21773752547780775, 0.14836105951525505, 0.0822611187186753, 0.040280922488712935, 0.01772939065670603,\
.007606057730113752, 0.004234626317954002, 0.00229582728870932, 0.0009174270464724253])
repetition_probs = repetition_probs/sum(repetition_probs)
print(len(rep_lengths)==len(repetition_probs))


for zbior in range(1):
    i=1
    with open("generowane_introny_repeatedends_%s.fasta" %zbior, 'w') as test_fasta:
        test_fasta.write(";nazwa - nrzbioru;10-(60+dl. powtorzenia);wersja=0;wersja konw;wersja niekonw\n")
        test_fasta.write(";seq - exon[-10:] intron[:25] intron[-25:] exon[:10]\n")
        while i<=1000000:
            prev_seq = "".join(np.random.choice(["A", "T", "C", "G"], 10, p=[at_exon, at_exon, gc_exon, gc_exon]))
            intron_seq = "".join(np.random.choice(["A", "T", "C", "G"], 50, p=[at_intron, at_intron, gc_intron, gc_intron]))
            rep_len = np.random.choice(rep_lengths, 1, p=repetition_probs)[0]
            repeated_end = intron_seq[0:rep_len]
            next_seq = "".join(np.random.choice(["A", "T", "C", "G"], 10, p=[at_exon, at_exon, gc_exon, gc_exon]))
            complete_seq = prev_seq + intron_seq + repeated_end + next_seq
            #print(len(complete_seq))
            scaffold_name = "0"
            prev_exon, next_exon = introns.Exon(scaffold_name=scaffold_name, scaffold_start=0, scaffold_end=10, sequence=prev_seq), introns.Exon(scaffold_name=scaffold_name, scaffold_start=60+len(repeated_end), scaffold_end=60+len(repeated_end)+10, sequence=next_seq)
            created_intron = introns.Intron(scaffold_name=scaffold_name, scaffold_start=10, scaffold_end=60+len(repeated_end), sequence=intron_seq+repeated_end, prev_exon=prev_exon, next_exon=next_exon)
            created_intron.conventional_version()
            created_intron.nonconventional_version()
            test_fasta.write(">%s;%s-%s;0;%s;%s\n%s\n" %(zbior,created_intron.scaffold_start, created_intron.scaffold_end, created_intron.best_conv_var, created_intron.best_nonconv_var, complete_seq))
            i+=1
print('done')

True
done


## Sztuczne introny do klasyfikatora

In [20]:
gc_intron, at_intron = .5312/2, (1-.5312)/2

f = open('test.fasta', 'w')

for i in range(200):
    intron_len = poisson.rvs(797, size=1)
    intron_seq = "".join(np.random.choice(["A", "T", "C", "G"], intron_len, p=[at_intron, at_intron, gc_intron, gc_intron]))
    f.write(">random intron %d\n" %i)
    f.write(intron_seq)
    f.write("\n")
f.close()
    
